In [0]:
from pyspark.sql import functions as F

customer_cdc_df = spark.table(
    "customer360_dev.control.customer_cdc_work"
)

In [0]:
latest_processed = (
    customer_cdc_df
    .agg(
        F.max("updated_at").alias(
            "last_processed_value"
        ),
        F.max("_source_file").alias(
            "last_processed_file"
        )
    )
    .first()
)

last_processed_value = (
    latest_processed["last_processed_value"]
)

last_processed_file = (
    latest_processed["last_processed_file"]
)

print(
    "Last processed timestamp:",
    last_processed_value
)

print(
    "Last processed file:",
    last_processed_file
)

Last processed timestamp: None
Last processed file: None


In [0]:
reconciliation_status = "PASS"
display(reconciliation_status)

'PASS'

In [0]:
if reconciliation_status == "PASS":

    watermark_update_df = spark.createDataFrame(
        [
            (
                "customers",
                "CRM",
                last_processed_value,
                last_processed_file,
                None,
                "SUCCESS"
            )
        ],
        """
        entity_name STRING,
        source_system STRING,
        last_processed_value TIMESTAMP,
        last_processed_file STRING,
        last_run_id STRING,
        load_status STRING
        """
    ).withColumn(
        "updated_at",
        F.current_timestamp()
    )

    watermark_update_df.createOrReplaceTempView(
        "customer_watermark_update"
    )

    spark.sql("""
        MERGE INTO customer360_dev.control.watermark AS tgt

        USING customer_watermark_update AS src

        ON tgt.entity_name = src.entity_name

        WHEN MATCHED THEN
        UPDATE SET
            tgt.source_system =
                src.source_system,

            tgt.last_processed_value =
                src.last_processed_value,

            tgt.last_processed_file =
                src.last_processed_file,

            tgt.last_run_id =
                src.last_run_id,

            tgt.load_status =
                src.load_status,

            tgt.updated_at =
                src.updated_at

        WHEN NOT MATCHED THEN
        INSERT (
            entity_name,
            source_system,
            last_processed_value,
            last_processed_file,
            last_run_id,
            load_status,
            updated_at
        )

        VALUES (
            src.entity_name,
            src.source_system,
            src.last_processed_value,
            src.last_processed_file,
            src.last_run_id,
            src.load_status,
            src.updated_at
        )
    """)

    print(
        "Customer watermark updated successfully."
    )

else:

    print(
        "Reconciliation failed. "
        "Watermark NOT updated."
    )

Customer watermark updated successfully.


In [0]:
%sql
SELECT
    entity_name,
    source_system,
    last_processed_value,
    last_processed_file,
    last_run_id,
    load_status,
    updated_at
FROM customer360_dev.control.watermark
WHERE entity_name = 'customers';

entity_name,source_system,last_processed_value,last_processed_file,last_run_id,load_status,updated_at
customers,CRM,null,null,null,SUCCESS,2026-09-27T11:27:19.887Z
